In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
import torch

In [ ]:
data = pd.read_csv("../../data/processed/model_df.csv")

In [ ]:
data

In [ ]:
data.columns

### LSTM-NARX

In [ ]:
# ============================================================
# 1. Import
# ============================================================
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
SEED = 42

In [ ]:
data["visit_date"] = pd.to_datetime(
    data["visit_date"],
    errors="coerce"
)

data["target_date"] = pd.to_datetime(
    data["target_date"],
    errors="coerce"
)

data = (
    data
    .sort_values(["patient_id", "visit_date"])
    .reset_index(drop=True)
)

print(data.shape)
data.head()

In [ ]:
# 由於blood資料缺失值較多，先不放
target_col = "target_mmse_1y"

exog_features = [
    # CDR
    "cdr_global",
    "cdr_memory",
    "cdr_orientation",
    "cdr_judgment",
    "cdr_community",
    "cdr_home_hobbies",
    "cdr_personal_care",
    "cdr_sob",

    # demographic / clinical
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia",

    # APOE
    "apoe4_carrier",

    # CASI
    "casi_mental_manipulation",
    "casi_attention",
    "casi_orientation",
    "casi_long_term_memory",
    "casi_short_term_memory",
    "casi_abstraction",
    "casi_drawing",
    "casi_verbal_fluency",
    "casi_language",
    "casi_total",

    "days_since_casi",
    "has_recent_casi",
]

In [ ]:
missing_cols = [
    col
    for col in exog_features + [target_col]
    if col not in data.columns
]

print("不存在欄位:", missing_cols)

In [ ]:
#只保留有 target 的 index visits
model_data = data[
    data[target_col].notna()
].copy()

print("samples:", len(model_data))
print("patients:", model_data["patient_id"].nunique())

In [ ]:
# Patient-level split
patient_ids = model_data[
    "patient_id"
].unique()

train_patients, temp_patients = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=SEED
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED
)

print("Train patients:", len(train_patients))
print("Val patients:", len(val_patients))
print("Test patients:", len(test_patients))

In [ ]:
#確認沒有patient leakage
print(
    len(set(train_patients) & set(val_patients)),
    len(set(train_patients) & set(test_patients)),
    len(set(val_patients) & set(test_patients))
)

In [ ]:
#對每一筆樣本，抓出這個病人在該次門診以前的 MMSE 歷史，
# 最多保留最近 7 次，當作輸入序列；再把這筆樣本對應的 1 年後 MMSE 當成預測目標。
MAX_LEN = 7

def build_sequences(
    full_data,
    sample_data,
    exog_features,
    max_len=7
):
    sequences = []
    sequence_lengths = []

    exogs = []
    targets = []

    patient_list = []
    index_dates = []

    patient_history = {
        pid: g.sort_values("visit_date").copy()
        for pid, g in full_data.groupby("patient_id")
    }

    for _, row in sample_data.iterrows():

        pid = row["patient_id"]
        index_date = row["visit_date"]

        history = patient_history[pid]

        history = history[
            history["visit_date"] <= index_date
        ]

        mmse_history = (
            history["mmse"]
            .dropna()
            .values
            .astype(float)
        )

        if len(mmse_history) == 0:
            continue

        mmse_history = mmse_history[-max_len:]

        sequences.append(mmse_history)
        sequence_lengths.append(
            len(mmse_history)
        )

        exogs.append(
            row[exog_features].values
        )

        targets.append(
            row["target_mmse_1y"]
        )

        patient_list.append(pid)
        index_dates.append(index_date)

    return (
        sequences,
        np.array(sequence_lengths),
        np.array(exogs, dtype=object),
        np.array(targets, dtype=float),
        np.array(patient_list),
        np.array(index_dates)
    )

#建立 train / val / test sample
train_samples = model_data[
    model_data["patient_id"].isin(train_patients)
].copy()

val_samples = model_data[
    model_data["patient_id"].isin(val_patients)
].copy()

test_samples = model_data[
    model_data["patient_id"].isin(test_patients)
].copy()

(
    train_seq_raw,
    train_lengths,
    X_train_exog_raw,
    y_train,
    train_pid,
    train_dates
) = build_sequences(
    data,
    train_samples,
    exog_features,
    MAX_LEN
)

(
    val_seq_raw,
    val_lengths,
    X_val_exog_raw,
    y_val,
    val_pid,
    val_dates
) = build_sequences(
    data,
    val_samples,
    exog_features,
    MAX_LEN
)

(
    test_seq_raw,
    test_lengths,
    X_test_exog_raw,
    y_test,
    test_pid,
    test_dates
) = build_sequences(
    data,
    test_samples,
    exog_features,
    MAX_LEN
)

print("train:", len(train_seq_raw))
print("val:", len(val_seq_raw))
print("test:", len(test_seq_raw))

In [ ]:
#Impute exogenous features
#血液目前沒放，所以這裡只處理 clinical/CASI。

X_train_exog_raw = X_train_exog_raw.astype(float)
X_val_exog_raw = X_val_exog_raw.astype(float)
X_test_exog_raw = X_test_exog_raw.astype(float)

exog_imputer = SimpleImputer(
    strategy="median"
)

X_train_exog = exog_imputer.fit_transform(
    X_train_exog_raw
)

X_val_exog = exog_imputer.transform(
    X_val_exog_raw
)

X_test_exog = exog_imputer.transform(
    X_test_exog_raw
)


#Standardize exogenous features

exog_scaler = StandardScaler()

X_train_exog = exog_scaler.fit_transform(
    X_train_exog
)

X_val_exog = exog_scaler.transform(
    X_val_exog
)

X_test_exog = exog_scaler.transform(
    X_test_exog
)

#Standardize MMSE sequences
train_mmse_values = np.concatenate(
    train_seq_raw
).reshape(-1, 1)

mmse_scaler = StandardScaler()

mmse_scaler.fit(
    train_mmse_values
)

In [ ]:
def scale_and_pad_sequences(
    sequences,
    scaler,
    max_len=7
):
    X = np.zeros(
        (len(sequences), max_len, 1),
        dtype=np.float32
    )

    lengths = []

    for i, seq in enumerate(sequences):

        seq = np.asarray(
            seq,
            dtype=float
        ).reshape(-1, 1)

        seq_scaled = (
            scaler
            .transform(seq)
            .flatten()
        )

        length = len(seq_scaled)

        X[
            i,
            :length,
            0
        ] = seq_scaled

        lengths.append(length)

    return (
        X,
        np.array(lengths)
    )

X_train_seq, train_lengths = \
    scale_and_pad_sequences(
        train_seq_raw,
        mmse_scaler,
        MAX_LEN
    )

X_val_seq, val_lengths = \
    scale_and_pad_sequences(
        val_seq_raw,
        mmse_scaler,
        MAX_LEN
    )

X_test_seq, test_lengths = \
    scale_and_pad_sequences(
        test_seq_raw,
        mmse_scaler,
        MAX_LEN
    )

In [ ]:
#Standardize target

y_scaler = StandardScaler()

y_train_scaled = (
    y_scaler
    .fit_transform(
        y_train.reshape(-1, 1)
    )
    .flatten()
)

y_val_scaled = (
    y_scaler
    .transform(
        y_val.reshape(-1, 1)
    )
    .flatten()
)

y_test_scaled = (
    y_scaler
    .transform(
        y_test.reshape(-1, 1)
    )
    .flatten()
)

In [ ]:
#建立 PyTorch Dataset

class MMSEDataset(Dataset):

    def __init__(
        self,
        X_seq,
        lengths,
        X_exog,
        y
    ):
        self.X_seq = torch.tensor(
            X_seq,
            dtype=torch.float32
        )

        self.lengths = torch.tensor(
            lengths,
            dtype=torch.long
        )

        self.X_exog = torch.tensor(
            X_exog,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.lengths[idx],
            self.X_exog[idx],
            self.y[idx]
        )

In [ ]:
#DataLoader

BATCH_SIZE = 32

train_dataset = MMSEDataset(
    X_train_seq,
    train_lengths,
    X_train_exog,
    y_train_scaled
)

val_dataset = MMSEDataset(
    X_val_seq,
    val_lengths,
    X_val_exog,
    y_val_scaled
)

test_dataset = MMSEDataset(
    X_test_seq,
    test_lengths,
    X_test_exog,
    y_test_scaled
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
#LSTM-NARX model

class LSTMNARX(nn.Module):

    def __init__(
        self,
        exog_dim,
        lstm_hidden=32,
        exog_hidden=32,
        fusion_hidden=32,
        dropout=0.2
    ):
        super().__init__()

        # --------------------------------------
        # MMSE autoregressive branch
        # --------------------------------------
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=lstm_hidden,
            batch_first=True
        )

        self.seq_dropout = nn.Dropout(
            dropout
        )

        # --------------------------------------
        # Exogenous branch
        # --------------------------------------
        self.exog_net = nn.Sequential(
            nn.Linear(
                exog_dim,
                exog_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # --------------------------------------
        # Fusion
        # --------------------------------------
        self.fusion = nn.Sequential(
            nn.Linear(
                lstm_hidden + exog_hidden,
                fusion_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(
                fusion_hidden,
                1
            )
        )

    def forward(
        self,
        x_seq,
        lengths,
        x_exog
    ):

        # pack padded sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            x_seq,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (h_n, _) = self.lstm(
            packed
        )

        # final hidden state
        seq_feat = h_n[-1]

        seq_feat = self.seq_dropout(
            seq_feat
        )

        # exogenous branch
        exog_feat = self.exog_net(
            x_exog
        )

        # concatenate
        fused = torch.cat(
            [
                seq_feat,
                exog_feat
            ],
            dim=1
        )

        output = self.fusion(
            fused
        )

        return output.squeeze(1)

# 初始化
model = LSTMNARX(
    exog_dim=X_train_exog.shape[1],
    lstm_hidden=32,
    exog_hidden=32,
    fusion_hidden=32,
    dropout=0.2
).to(device)

print(model)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    for (
        x_seq,
        lengths,
        x_exog,
        y
    ) in loader:

        x_seq = x_seq.to(device)
        lengths = lengths.to(device)
        x_exog = x_exog.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred = model(
            x_seq,
            lengths,
            x_exog
        )

        loss = criterion(
            pred,
            y
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * len(y)
        )

    return (
        total_loss
        / len(loader.dataset)
    )

def evaluate_loss(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for (
            x_seq,
            lengths,
            x_exog,
            y
        ) in loader:

            x_seq = x_seq.to(device)
            lengths = lengths.to(device)
            x_exog = x_exog.to(device)
            y = y.to(device)

            pred = model(
                x_seq,
                lengths,
                x_exog
            )

            loss = criterion(
                pred,
                y
            )

            total_loss += (
                loss.item()
                * len(y)
            )

    return (
        total_loss
        / len(loader.dataset)
    )

In [ ]:
#Early stopping training loop

MAX_EPOCHS = 100
PATIENCE = 10

best_val_loss = np.inf
best_state = None

patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate_loss(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(
        train_loss
    )

    val_losses.append(
        val_loss
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:
            print(
                "Early stopping."
            )
            break

model.load_state_dict(
    best_state
)

model = model.to(device)

In [ ]:
def predict_model(
    model,
    loader,
    device
):

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for (
            x_seq,
            lengths,
            x_exog,
            y
        ) in loader:

            x_seq = x_seq.to(device)
            lengths = lengths.to(device)
            x_exog = x_exog.to(device)

            pred = model(
                x_seq,
                lengths,
                x_exog
            )

            preds.extend(
                pred.cpu().numpy()
            )

            targets.extend(
                y.numpy()
            )

    return (
        np.array(preds),
        np.array(targets)
    )

y_pred_scaled, _ = predict_model(
    model,
    test_loader,
    device
)

#反標準化
y_pred = (
    y_scaler
    .inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    )
    .flatten()
)

In [ ]:
#評估

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)

bias = np.mean(
    y_pred - y_test
)

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")
print(f"Bias : {bias:.4f}")

In [ ]:
result_df = pd.DataFrame({
    "patient_id": test_pid,
    "index_date": test_dates,
    "true_mmse_1y": y_test,
    "pred_mmse_1y": y_pred
})

result_df["error"] = (
    result_df["pred_mmse_1y"]
    - result_df["true_mmse_1y"]
)

result_df.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

plt.plot(
    train_losses,
    label="Train Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

### xgboost

In [ ]:
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import numpy as np
import pandas as pd

In [ ]:
xgb_features = [
    # Current + longitudinal MMSE
    "mmse",
    "previous_mmse",
    "mmse_change_prev",
    "days_since_prev_mmse",

    # CDR
    "cdr_global",
    "cdr_memory",
    "cdr_orientation",
    "cdr_judgment",
    "cdr_community",
    "cdr_home_hobbies",
    "cdr_personal_care",
    "cdr_sob",

    # Demographic / clinical
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia",

    # APOE
    "apoe4_carrier",

    # CASI
    "casi_mental_manipulation",
    "casi_attention",
    "casi_orientation",
    "casi_long_term_memory",
    "casi_short_term_memory",
    "casi_abstraction",
    "casi_drawing",
    "casi_verbal_fluency",
    "casi_language",
    "casi_total",

    "days_since_casi",
    "has_recent_casi",
]

target_col = "target_mmse_1y"

In [ ]:
xgb_train = model_data[
    model_data["patient_id"].isin(train_patients)
].copy()

xgb_val = model_data[
    model_data["patient_id"].isin(val_patients)
].copy()

xgb_test = model_data[
    model_data["patient_id"].isin(test_patients)
].copy()

In [ ]:
X_train_xgb = xgb_train[xgb_features].copy()
y_train_xgb = xgb_train[target_col].copy()

X_val_xgb = xgb_val[xgb_features].copy()
y_val_xgb = xgb_val[target_col].copy()

X_test_xgb = xgb_test[xgb_features].copy()
y_test_xgb = xgb_test[target_col].copy()

print("Train:", X_train_xgb.shape)
print("Val  :", X_val_xgb.shape)
print("Test :", X_test_xgb.shape)

print("\nPatients:")
print("Train:", xgb_train["patient_id"].nunique())
print("Val  :", xgb_val["patient_id"].nunique())
print("Test :", xgb_test["patient_id"].nunique())

In [ ]:
X_train_xgb = X_train_xgb.apply(
    pd.to_numeric,
    errors="coerce"
)

X_val_xgb = X_val_xgb.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test_xgb = X_test_xgb.apply(
    pd.to_numeric,
    errors="coerce"
)

pd.DataFrame({
    "missing_n": X_train_xgb.isna().sum(),
    "missing_rate": X_train_xgb.isna().mean()
}).sort_values(
    "missing_rate",
    ascending=False
)

In [ ]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",

    n_estimators=1000,
    learning_rate=0.03,

    max_depth=4,
    min_child_weight=5,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=SEED,
    n_jobs=-1,

    eval_metric="rmse"
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb,

    eval_set=[
        (X_train_xgb, y_train_xgb),
        (X_val_xgb, y_val_xgb)
    ],

    verbose=False
)

In [ ]:
y_pred_xgb = xgb_model.predict(
    X_test_xgb
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test_xgb,
        y_pred_xgb
    )
)

xgb_mae = mean_absolute_error(
    y_test_xgb,
    y_pred_xgb
)

xgb_r2 = r2_score(
    y_test_xgb,
    y_pred_xgb
)

xgb_bias = np.mean(
    y_pred_xgb - y_test_xgb
)

print(f"XGBoost RMSE : {xgb_rmse:.4f}")
print(f"XGBoost MAE  : {xgb_mae:.4f}")
print(f"XGBoost R²   : {xgb_r2:.4f}")
print(f"XGBoost Bias : {xgb_bias:.4f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Naive",
        "LSTM-NARX",
        "XGBoost"
    ],

    "RMSE": [
        2.6878,
        2.4318,
        xgb_rmse
    ],

    "MAE": [
        1.8948,
        1.7377,
        xgb_mae
    ],

    "R2": [
        0.8802,
        0.9019,
        xgb_r2
    ],

    "Bias": [
        0.7877,
        0.0967,
        xgb_bias
    ]
})

comparison

In [ ]:
importance_df = pd.DataFrame({
    "Feature": xgb_features,
    "Importance": xgb_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

importance_df.head(15)

### TCN 

In [ ]:
import torch
import torch.nn as nn
import numpy as np

In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size]


class TemporalBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout=0.2
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding
        )

        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding
        )

        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1,
            self.chomp1,
            self.relu1,
            self.dropout1,
            self.conv2,
            self.chomp2,
            self.relu2,
            self.dropout2
        )

        if in_channels != out_channels:
            self.downsample = nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.downsample = None

        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)

        residual = (
            x if self.downsample is None
            else self.downsample(x)
        )

        return self.final_relu(out + residual)

class TCNClinicalModel(nn.Module):
    def __init__(
        self,
        exog_dim,
        tcn_channels=32,
        kernel_size=3,
        exog_hidden=32,
        fusion_hidden=32,
        dropout=0.2
    ):
        super().__init__()

        self.tcn = nn.Sequential(
            TemporalBlock(
                in_channels=1,
                out_channels=tcn_channels,
                kernel_size=kernel_size,
                dilation=1,
                dropout=dropout
            ),
            TemporalBlock(
                in_channels=tcn_channels,
                out_channels=tcn_channels,
                kernel_size=kernel_size,
                dilation=2,
                dropout=dropout
            )
        )

        self.seq_dropout = nn.Dropout(dropout)

        self.exog_net = nn.Sequential(
            nn.Linear(exog_dim, exog_hidden),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                tcn_channels + exog_hidden,
                fusion_hidden
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1)
        )

    def forward(
        self,
        x_seq,
        lengths,
        x_exog
    ):
        # [batch, time, feature]
        # ->
        # [batch, feature, time]
        x_seq = x_seq.transpose(1, 2)

        tcn_out = self.tcn(x_seq)

        batch_size = tcn_out.size(0)

        last_index = (
            lengths.to(tcn_out.device) - 1
        )

        batch_index = torch.arange(
            batch_size,
            device=tcn_out.device
        )

        seq_feat = tcn_out[
            batch_index,
            :,
            last_index
        ]

        seq_feat = self.seq_dropout(seq_feat)

        exog_feat = self.exog_net(x_exog)

        fused = torch.cat(
            [seq_feat, exog_feat],
            dim=1
        )

        output = self.fusion(fused)

        return output.squeeze(1)

In [ ]:
tcn_model = TCNClinicalModel(

    exog_dim=X_train_exog.shape[1],

    tcn_channels=32,

    kernel_size=3,

    exog_hidden=32,

    fusion_hidden=32,

    dropout=0.2

).to(device)

print(tcn_model)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    tcn_model.parameters(),
    lr=1e-3
)

MAX_EPOCHS = 100
PATIENCE = 10

best_val_loss = np.inf
best_state = None

patience_counter = 0

tcn_train_losses = []
tcn_val_losses = []

#訓練
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    train_loss = train_one_epoch(
        tcn_model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate_loss(
        tcn_model,
        val_loader,
        criterion,
        device
    )

    tcn_train_losses.append(
        train_loss
    )

    tcn_val_losses.append(
        val_loss
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    # ==========================================
    # Early stopping
    # ==========================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            k: v.cpu().clone()
            for k, v
            in tcn_model.state_dict().items()
        }

        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("Early stopping.")

            break

In [ ]:
tcn_model.load_state_dict(
    best_state
)

tcn_model = tcn_model.to(
    device
)

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

plt.plot(
    tcn_train_losses,
    label="Train Loss"
)

plt.plot(
    tcn_val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title(
    "TCN Training and Validation Loss"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
y_pred_tcn_scaled, _ = predict_model(
    tcn_model,
    test_loader,
    device
)

y_pred_tcn = (
    y_scaler
    .inverse_transform(
        y_pred_tcn_scaled.reshape(-1, 1)
    )
    .flatten()
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

tcn_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tcn
    )
)

tcn_mae = mean_absolute_error(
    y_test,
    y_pred_tcn
)

tcn_r2 = r2_score(
    y_test,
    y_pred_tcn
)

tcn_bias = np.mean(
    y_pred_tcn - y_test
)

print(
    f"TCN RMSE : {tcn_rmse:.4f}"
)

print(
    f"TCN MAE  : {tcn_mae:.4f}"
)

print(
    f"TCN R²   : {tcn_r2:.4f}"
)

print(
    f"TCN Bias : {tcn_bias:.4f}"
)

### ode-lstm

In [ ]:
MAX_LEN = 7

def build_ode_sequences(
    full_data,
    sample_data,
    exog_features,
    max_len=7
):
    sequences = []
    time_gaps = []
    sequence_lengths = []

    exogs = []
    targets = []

    patient_list = []
    index_dates = []

    patient_history = {
        pid: g.sort_values("visit_date").copy()
        for pid, g in full_data.groupby("patient_id")
    }

    for _, row in sample_data.iterrows():

        pid = row["patient_id"]
        index_date = row["visit_date"]

        history = patient_history[pid].copy()

        # 僅能使用 index date 當下或以前資料
        history = history[
            history["visit_date"] <= index_date
        ].copy()

        # MMSE 缺失的 observation 不參與 sequence
        history = history[
            history["mmse"].notna()
        ].copy()

        if len(history) == 0:
            continue

        # 最多取最近 7 次
        history = history.tail(max_len).copy()

        mmse_seq = (
            history["mmse"]
            .astype(float)
            .values
        )

        dates = pd.to_datetime(
            history["visit_date"]
        )

        # 相鄰 observation 時間差
        delta_days = (
            dates.diff()
            .dt.days
            .fillna(0)
            .values
            .astype(float)
        )

        # 改成年
        delta_years = (
            delta_days / 365.25
        )

        sequences.append(mmse_seq)
        time_gaps.append(delta_years)

        sequence_lengths.append(
            len(mmse_seq)
        )

        exogs.append(
            row[exog_features].values
        )

        targets.append(
            row["target_mmse_1y"]
        )

        patient_list.append(pid)
        index_dates.append(index_date)

    return (
        sequences,
        time_gaps,
        np.array(sequence_lengths),
        np.array(exogs, dtype=object),
        np.array(targets, dtype=float),
        np.array(patient_list),
        np.array(index_dates)
    )

In [ ]:
(
    train_ode_seq_raw,
    train_dt_raw,
    train_ode_lengths,
    X_train_ode_exog_raw,
    y_train_ode,
    train_ode_pid,
    train_ode_dates
) = build_ode_sequences(
    data,
    train_samples,
    exog_features,
    MAX_LEN
)


(
    val_ode_seq_raw,
    val_dt_raw,
    val_ode_lengths,
    X_val_ode_exog_raw,
    y_val_ode,
    val_ode_pid,
    val_ode_dates
) = build_ode_sequences(
    data,
    val_samples,
    exog_features,
    MAX_LEN
)


(
    test_ode_seq_raw,
    test_dt_raw,
    test_ode_lengths,
    X_test_ode_exog_raw,
    y_test_ode,
    test_ode_pid,
    test_ode_dates
) = build_ode_sequences(
    data,
    test_samples,
    exog_features,
    MAX_LEN
)

print("train:", len(train_ode_seq_raw))
print("val  :", len(val_ode_seq_raw))
print("test :", len(test_ode_seq_raw))

In [ ]:
def scale_pad_ode_sequences(
    sequences,
    time_gaps,
    scaler,
    max_len=7
):

    X_seq = np.zeros(
        (len(sequences), max_len, 1),
        dtype=np.float32
    )

    X_dt = np.zeros(
        (len(sequences), max_len),
        dtype=np.float32
    )

    lengths = []

    for i, (seq, dt) in enumerate(
        zip(sequences, time_gaps)
    ):

        seq = np.asarray(
            seq,
            dtype=float
        ).reshape(-1, 1)

        seq_scaled = (
            scaler.transform(seq)
            .flatten()
        )

        dt = np.asarray(
            dt,
            dtype=np.float32
        )

        length = len(seq_scaled)

        X_seq[
            i,
            :length,
            0
        ] = seq_scaled

        X_dt[
            i,
            :length
        ] = dt

        lengths.append(length)

    return (
        X_seq,
        X_dt,
        np.array(lengths)
    )

X_train_ode_seq, X_train_dt, train_ode_lengths = \
    scale_pad_ode_sequences(
        train_ode_seq_raw,
        train_dt_raw,
        mmse_scaler,
        MAX_LEN
    )

X_val_ode_seq, X_val_dt, val_ode_lengths = \
    scale_pad_ode_sequences(
        val_ode_seq_raw,
        val_dt_raw,
        mmse_scaler,
        MAX_LEN
    )

X_test_ode_seq, X_test_dt, test_ode_lengths = \
    scale_pad_ode_sequences(
        test_ode_seq_raw,
        test_dt_raw,
        mmse_scaler,
        MAX_LEN
    )

#Exogenous variables 也沿用原本 preprocessing
X_train_ode_exog_raw = (
    X_train_ode_exog_raw.astype(float)
)

X_val_ode_exog_raw = (
    X_val_ode_exog_raw.astype(float)
)

X_test_ode_exog_raw = (
    X_test_ode_exog_raw.astype(float)
)

#用你前面的 imputer
X_train_ode_exog = exog_imputer.transform(
    X_train_ode_exog_raw
)

X_val_ode_exog = exog_imputer.transform(
    X_val_ode_exog_raw
)

X_test_ode_exog = exog_imputer.transform(
    X_test_ode_exog_raw
)

#再用原本 scaler
X_train_ode_exog = exog_scaler.transform(
    X_train_ode_exog
)

X_val_ode_exog = exog_scaler.transform(
    X_val_ode_exog
)

X_test_ode_exog = exog_scaler.transform(
    X_test_ode_exog
)

#Target 也沿用
y_train_ode_scaled = (
    y_scaler.transform(
        y_train_ode.reshape(-1, 1)
    ).flatten()
)

y_val_ode_scaled = (
    y_scaler.transform(
        y_val_ode.reshape(-1, 1)
    ).flatten()
)

y_test_ode_scaled = (
    y_scaler.transform(
        y_test_ode.reshape(-1, 1)
    ).flatten()
)

In [ ]:
#建立 ODE Dataset

class ODEMMSEDataset(Dataset):

    def __init__(
        self,
        X_seq,
        X_dt,
        lengths,
        X_exog,
        y
    ):

        self.X_seq = torch.tensor(
            X_seq,
            dtype=torch.float32
        )

        self.X_dt = torch.tensor(
            X_dt,
            dtype=torch.float32
        )

        self.lengths = torch.tensor(
            lengths,
            dtype=torch.long
        )

        self.X_exog = torch.tensor(
            X_exog,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):

        return (
            self.X_seq[idx],
            self.X_dt[idx],
            self.lengths[idx],
            self.X_exog[idx],
            self.y[idx]
        )

#dataloader
BATCH_SIZE = 32

train_ode_dataset = ODEMMSEDataset(
    X_train_ode_seq,
    X_train_dt,
    train_ode_lengths,
    X_train_ode_exog,
    y_train_ode_scaled
)

val_ode_dataset = ODEMMSEDataset(
    X_val_ode_seq,
    X_val_dt,
    val_ode_lengths,
    X_val_ode_exog,
    y_val_ode_scaled
)

test_ode_dataset = ODEMMSEDataset(
    X_test_ode_seq,
    X_test_dt,
    test_ode_lengths,
    X_test_ode_exog,
    y_test_ode_scaled
)

train_ode_loader = DataLoader(
    train_ode_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ode_loader = DataLoader(
    val_ode_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ode_loader = DataLoader(
    test_ode_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
class ODEFunc(nn.Module):

    def __init__(
        self,
        hidden_dim=64
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.Tanh(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )

    def forward(self, h):

        return self.net(h)

#簡化式 ODE-LSTM
class ODELSTMClinicalModel(nn.Module):

    def __init__(
        self,
        exog_dim,
        hidden_dim=64,
        exog_hidden=64,
        fusion_hidden=64,
        dropout=0.2
    ):

        super().__init__()

        self.hidden_dim = hidden_dim

        # ====================================
        # Continuous-time ODE function
        # ====================================

        self.ode_func = ODEFunc(
            hidden_dim=hidden_dim
        )

        # ====================================
        # Observation update
        # ====================================

        self.lstm_cell = nn.LSTMCell(
            input_size=1,
            hidden_size=hidden_dim
        )

        # ====================================
        # Exogenous branch
        # ====================================

        self.exog_net = nn.Sequential(

            nn.Linear(
                exog_dim,
                exog_hidden
            ),

            nn.ReLU(),

            nn.Dropout(dropout)
        )

        # ====================================
        # Fusion
        # ====================================

        self.fusion = nn.Sequential(

            nn.Linear(
                hidden_dim + exog_hidden,
                fusion_hidden
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                fusion_hidden,
                1
            )
        )

    def forward(
        self,
        x_seq,
        delta_t,
        lengths,
        x_exog
    ):

        batch_size = x_seq.size(0)

        device = x_seq.device

        # initial states
        h = torch.zeros(
            batch_size,
            self.hidden_dim,
            device=device
        )

        c = torch.zeros(
            batch_size,
            self.hidden_dim,
            device=device
        )

        max_len = x_seq.size(1)

        for t in range(max_len):

            # 哪些 patient 在此 timestep 仍有效
            active = (
                t < lengths
            ).float().unsqueeze(1)

            dt = (
                delta_t[:, t]
                .unsqueeze(1)
            )

            # ======================================
            # 1. ODE evolution between observations
            # ======================================

            dh = self.ode_func(h)

            h_ode = (
                h + dt * dh
            )

            # ======================================
            # 2. LSTM observation update
            # ======================================

            x_t = x_seq[:, t, :]

            h_new, c_new = self.lstm_cell(
                x_t,
                (h_ode, c)
            )

            # padding timestep 不更新 state
            h = (
                active * h_new
                + (1 - active) * h
            )

            c = (
                active * c_new
                + (1 - active) * c
            )

        # ======================================
        # Clinical / exogenous branch
        # ======================================

        exog_feat = self.exog_net(
            x_exog
        )

        # ======================================
        # Fusion
        # ======================================

        fused = torch.cat(
            [
                h,
                exog_feat
            ],
            dim=1
        )

        output = self.fusion(
            fused
        )

        return output.squeeze(1)

In [ ]:
ode_lstm_model = ODELSTMClinicalModel(

    exog_dim=X_train_ode_exog.shape[1],

    hidden_dim=64,

    exog_hidden=64,

    fusion_hidden=64,

    dropout=0.2

).to(device)

print(ode_lstm_model)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    ode_lstm_model.parameters(),
    lr=0.001
)

def train_ode_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    for (
        x_seq,
        x_dt,
        lengths,
        x_exog,
        y
    ) in loader:

        x_seq = x_seq.to(device)
        x_dt = x_dt.to(device)
        lengths = lengths.to(device)
        x_exog = x_exog.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred = model(
            x_seq,
            x_dt,
            lengths,
            x_exog
        )

        loss = criterion(
            pred,
            y
        )

        loss.backward()

        # ODE/RNN 建議限制 gradient
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        total_loss += (
            loss.item()
            * len(y)
        )

    return (
        total_loss
        / len(loader.dataset)
    )

def evaluate_ode_loss(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for (
            x_seq,
            x_dt,
            lengths,
            x_exog,
            y
        ) in loader:

            x_seq = x_seq.to(device)
            x_dt = x_dt.to(device)
            lengths = lengths.to(device)
            x_exog = x_exog.to(device)
            y = y.to(device)

            pred = model(
                x_seq,
                x_dt,
                lengths,
                x_exog
            )

            loss = criterion(
                pred,
                y
            )

            total_loss += (
                loss.item()
                * len(y)
            )

    return (
        total_loss
        / len(loader.dataset)
    )

In [ ]:
MAX_EPOCHS = 100
PATIENCE = 10

best_val_loss = np.inf
best_state = None

patience_counter = 0

ode_train_losses = []
ode_val_losses = []

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    train_loss = train_ode_epoch(
        ode_lstm_model,
        train_ode_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate_ode_loss(
        ode_lstm_model,
        val_ode_loader,
        criterion,
        device
    )

    ode_train_losses.append(
        train_loss
    )

    ode_val_losses.append(
        val_loss
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            k: v.cpu().clone()
            for k, v
            in ode_lstm_model.state_dict().items()
        }

        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("Early stopping.")

            break

ode_lstm_model.load_state_dict(
    best_state
)

ode_lstm_model = ode_lstm_model.to(
    device
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

plt.plot(
    ode_train_losses,
    label="Train Loss"
)

plt.plot(
    ode_val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.title(
    "ODE-LSTM Training and Validation Loss"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
def predict_ode_model(
    model,
    loader,
    device
):

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for (
            x_seq,
            x_dt,
            lengths,
            x_exog,
            y
        ) in loader:

            x_seq = x_seq.to(device)
            x_dt = x_dt.to(device)
            lengths = lengths.to(device)
            x_exog = x_exog.to(device)

            pred = model(
                x_seq,
                x_dt,
                lengths,
                x_exog
            )

            preds.extend(
                pred.cpu().numpy()
            )

            targets.extend(
                y.numpy()
            )

    return (
        np.array(preds),
        np.array(targets)
    )

y_pred_ode_scaled, _ = predict_ode_model(
    ode_lstm_model,
    test_ode_loader,
    device
)

y_pred_ode = (
    y_scaler.inverse_transform(
        y_pred_ode_scaled.reshape(-1, 1)
    )
    .flatten()
)

ode_rmse = np.sqrt(
    mean_squared_error(
        y_test_ode,
        y_pred_ode
    )
)

ode_mae = mean_absolute_error(
    y_test_ode,
    y_pred_ode
)

ode_r2 = r2_score(
    y_test_ode,
    y_pred_ode
)

ode_bias = np.mean(
    y_pred_ode - y_test_ode
)

print(
    f"ODE-LSTM RMSE : {ode_rmse:.4f}"
)

print(
    f"ODE-LSTM MAE  : {ode_mae:.4f}"
)

print(
    f"ODE-LSTM R²   : {ode_r2:.4f}"
)

print(
    f"ODE-LSTM Bias : {ode_bias:.4f}"
)

comparison = pd.DataFrame({
    "Model": [
        "Naive",
        "XGBoost",
        "LSTM-NARX",
        "TCN",
        "ODE-LSTM"
    ],

    "RMSE": [
        2.6878,
        2.527761,
        2.4318,
        2.4003,
        ode_rmse
    ],

    "MAE": [
        1.8948,
        1.770394,
        1.7377,
        1.6926,
        ode_mae
    ],

    "R2": [
        0.8802,
        0.894042,
        0.9019,
        0.9045,
        ode_r2
    ],

    "Bias": [
        0.7877,
        0.132948,
        0.0967,
        0.1681,
        ode_bias
    ]
})

comparison